# Дообучение распознавателя ANPR (прицепы)

Справа: **Accelerator → GPU T4 x2**, **Internet → On**, **Add Input** → свой приватный датасет `anpr-uz`. Затем «Run All».

Хватит одной T4: модель небольшая, вторая карта здесь почти ничего не даёт.

In [ ]:
import glob, os, shutil

# Kaggle распаковывает zip по-разному — ищем gt.txt, а не угадываем путь
GT = glob.glob('/kaggle/input/**/recog/gt.txt', recursive=True)[0]
DATA = os.path.dirname(GT)
CODE = glob.glob('/kaggle/input/**/code/pyproject.toml', recursive=True)[0]
# /kaggle/input только для чтения, а pip install -e пишет рядом с кодом
shutil.copytree(os.path.dirname(CODE), '/kaggle/working/anpr', dirs_exist_ok=True)
print('данные:', DATA)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working/anpr
!pip install -q pydantic-settings huggingface_hub
!pip install -q --no-deps -e .

from huggingface_hub import hf_hub_download
hf_hub_download('IzzatulloOne/anpr-uz', 'best_accuracy.pth', local_dir='/kaggle/working/anpr')
print('исходные веса на месте')

In [ ]:
# Около часа на T4. Лучшая эпоха сохраняется сразу — прерванный прогон не пропадает.
!python -m tools.train_recog --data {DATA} --out /kaggle/working/run \
    --epochs 30 --batch 128 --workers 4

In [ ]:
import json
log = json.load(open('/kaggle/working/run/metrics.json'))
for r in log:
    print(r['epoch'], r.get('loss', '-'), r['macro'], r['per_class'])

Готовый чекпоинт: **Output → run/best_accuracy.pth** — скачать и положить в корень проекта вместо старого (старый сохранить). Затем дома:

```
./.venv/bin/python -m tools.eval_uz --run           # однострочные не просели?
ANPR_TWO_LINE=true ANPR_TWO_LINE_MODE=unfold ...      # прицепы читаются?
```